# Encode a large line file and resume
Read a bounded number of lines at a time. Every line, including an empty line, owns one row.
The example interrupts after two batches, then resumes twice. It checks that no rows are duplicated.
The input must remain immutable. In production retain an input identity, encoder fingerprint and batch policy with the job.
Here the source and store live in one temporary job directory, and the same immutable source survives the simulated restart.
Use one writer. Keep the vector store on a supported local filesystem even when the input is on a network share.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import platform
from datetime import date

scratch = TemporaryDirectory(prefix="fse-recipe-")
work = Path(scratch.name)
# Third-party caches also belong to this temporary directory.
for name in ("HF_HOME", "XDG_CACHE_HOME", "NUMBA_CACHE_DIR", "MPLCONFIGDIR"):
    os.environ[name] = str(work / name.lower())
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import fse
print(date.today().isoformat(), platform.platform(), platform.machine())

In [ ]:
# Deliberately tiny, hand-built word rows. These demonstrate APIs, not model quality.
words = ["cat", "dog", "pet", "kitten", "puppy", "car", "bus", "road", "train", "truck"]
vectors = np.array([
    [1, 0, .2], [1, .1, 0], [1, 0, 0], [1, 0, .3], [1, .2, 0],
    [0, 1, .2], [0, 1, .1], [0, 1, 0], [0, 1, .3], [0, 1, .4],
], dtype=np.float32)
encoder = fse.Encoder.from_arrays(vectors, words=words, normalize=False)
texts = ["cat pet", "dog puppy", "car road", "bus truck"]

In [ ]:
from itertools import islice
source = work / "input.txt"
source.write_text("\n".join((texts + [""]) * 1000) + "\n", encoding="utf-8")
store_path = work / "store"

def line_batches(path, skip, size=700):
    with path.open(encoding="utf-8") as stream:
        for _ in islice(stream, skip):
            pass
        while batch := list(islice(stream, size)):
            yield [line.rstrip("\n") for line in batch]

def resume(stop_after=None):
    with fse.ShardSink(store_path, encoder, shard_rows=700) as sink:
        start = sink.first_row  # existing store prefix, not this run's durable count
        stream = encoder.encode_batches(line_batches(source, start), sink=sink, workers=1)
        try:
            for number, batch in enumerate(stream, 1):
                assert batch.durable_rows is not None
                # Only this prefix can be acknowledged before close.
                confirmed = start + batch.durable_rows
                if stop_after is not None and number == stop_after:
                    break
        finally:
            stream.close()
    with fse.VectorStore.open(store_path) as store:
        return len(store)

first = resume(stop_after=2)
second = resume()
third = resume()
assert first == 1400 and second == third == 5000
with fse.VectorStore.open(store_path) as store:
    np.testing.assert_allclose(store.score(np.eye(encoder.output_dim, dtype=np.float32), mode="exact", workers=1), encoder.encode(source, workers=1))
print({"interrupted_rows": first, "resumed_rows": second, "rerun_rows": third})

In [ ]:
scratch.cleanup()